# Gradients Real Training Demo

This notebook trains a small Qwen base model on the Gradients PubMedQA normalized training set, then compares answers from the base model and the trained model on a few held-out test prompts.

The aim is that the trained model will provide factually correct medical data and improved evidence over the base model.

You only need a Gradients API key to launch training. For loading a private trained model from Hugging Face, set `HF_TOKEN` in your environment before running the inference cells.

## Install Packages

Run this once. `%pip` installs into the same Python environment used by this notebook.

In [ ]:
%pip install -q --upgrade gradientsio==0.1.2

## Setup

Enter your Gradients API key when prompted. The defaults use the normalized PubMedQA train/test datasets, `Qwen/Qwen2.5-3B`, and a 2 hour training run.

In [ ]:
import os
import time
from getpass import getpass

from IPython.display import Markdown
from IPython.display import display
from gradientsio import GenerationConfig
from gradientsio import GradientsClient
from gradientsio import ModelSampler
from gradientsio import TaskType
from gradientsio import load_dataset_rows

MODEL_ID = "Qwen/Qwen2.5-3B"
TRAIN_DATASET = "gradients-io-tournaments/PubMedQA-Normalized-Train"
TEST_DATASET = "gradients-io-tournaments/PubMedQA-Normalized-Test"
HOURS_TO_TRAIN = 2
RESULT_MODEL_NAME = "pubmedqa-qwen2-5-3b-gradients-demo"
SAMPLE_SIZE = 5
SAMPLE_SEED = 23223
POLL_INTERVAL_SECONDS = 300
REQUIRE_CUDA = True
GENERATION = GenerationConfig(max_new_tokens=96, repetition_penalty=1.12, num_beams=4)

if not os.getenv("GRADIENTS_API_KEY"):
    os.environ["GRADIENTS_API_KEY"] = getpass("Gradients API key: ").strip()

client = GradientsClient()

print("Setup complete.")
print(f"Base model: {MODEL_ID}")
print(f"Train dataset: {TRAIN_DATASET}")
print(f"Test dataset: {TEST_DATASET}")

## Start Training

This creates an instruct fine-tuning task on Gradients. Save the printed task ID if you close the notebook; you can paste it into the wait cell later.

In [ ]:
task = client.train(
    model=MODEL_ID,
    task_type=TaskType.INSTRUCT,
    hours=HOURS_TO_TRAIN,
    dataset=TRAIN_DATASET,
    field_instruction="instruction",
    field_input="input",
    field_output="output",
    result_model_name=RESULT_MODEL_NAME,
)

TASK_ID = task.task_id
print(f"Training task created: {TASK_ID}")
print("You can now run the next cell to wait for training to finish.")

## Wait For Training

Run this after starting training. If you already have a task ID from a previous run, set `TASK_ID` before running this cell.

In [ ]:
if "TASK_ID" not in globals() or not TASK_ID:
    TASK_ID = input("Paste an existing Gradients task ID: ").strip()

training_task = client.tasks.handle(TASK_ID)

while True:
    details = training_task.refresh()
    status = details.status
    trained_repo = details.trained_model_repository
    print(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | status={status} | trained_model_repository={trained_repo}")

    if details.is_terminal:
        break

    time.sleep(POLL_INTERVAL_SECONDS)

if not details.is_success:
    raise RuntimeError(f"Training did not finish successfully. Final status: {details.status}")

TRAINED_MODEL_REPO = details.trained_model_repository
if not TRAINED_MODEL_REPO:
    raise RuntimeError("Training succeeded, but no trained_model_repository was returned.")

print(f"Training complete: {TRAINED_MODEL_REPO}")

## Pick Test Prompts

This samples a few held-out examples from the normalized PubMedQA test dataset. Each example has an instruction, abstracts in `input`, and the expected answer in `output`.

In [ ]:
samples = load_dataset_rows(TEST_DATASET, sample_size=SAMPLE_SIZE, seed=SAMPLE_SEED)


def build_prompt(row):
    instruction = (row.get("instruction") or "").strip()
    return f"{instruction}\n\nAnswer:"


for index, row in enumerate(samples, start=1):
    prompt = build_prompt(row)
    print(f"Example {index} | PubMed ID: {row.get('pubid')}")
    print(prompt.removesuffix("Answer:").strip()[:500])
    print(f"Expected: {(row.get('output') or '')[:250]}...")
    print()

## Compare Results

This cell keeps the PubMedQA-specific prompt and display format in the notebook. The SDK only handles dataset rows, model loading, adapter merging, and generation.

In [ ]:
if "TRAINED_MODEL_REPO" not in globals() or not TRAINED_MODEL_REPO:
    TRAINED_MODEL_REPO = input("Paste the trained_model_repository from Gradients: ").strip()

samples = load_dataset_rows(TEST_DATASET, sample_size=SAMPLE_SIZE, seed=SAMPLE_SEED)
prompts = [build_prompt(row) for row in samples]

sampler = ModelSampler(require_cuda=REQUIRE_CUDA)
print(sampler.cuda_status())

base_answers = sampler.generate(MODEL_ID, prompts, config=GENERATION)
trained_answers = sampler.generate_with_adapter(
    TRAINED_MODEL_REPO,
    prompts,
    base_model_repo=MODEL_ID,
    config=GENERATION,
)

sections = []
for index, row in enumerate(samples, start=1):
    question = prompts[index - 1].removesuffix("Answer:").strip()
    expected = (row.get("output") or "").strip()
    trained = trained_answers[index - 1].strip()
    base = base_answers[index - 1].strip()
    sections.append(
        f"""
---

## Example {index} | PubMed ID: {row.get('pubid')}

### Question

{question}

### Expected Answer

{expected}

### Trained Model Answer

{trained}

### Base Model Answer

{base}
"""
    )

display(Markdown("\n".join(sections)))